# Phase 1: Loading, Exploring and Cleaning the Datasets

In [10]:
import pandas as pd

# Loading data
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')

# Merging on TransactionID
train = pd.merge(train_trans, train_id, on='TransactionID', how='left')

print("Data successfully loaded and merged...")
print(f"Total rows: {len(train)}")
print(f"Train dataset shape: {train.shape}")

Data successfully loaded and merged...
Total rows: 590540
Train dataset shape: (590540, 434)


In [11]:
# Calculating the percentage of missing values for each feature
missing_percentages = (train.isnull().sum() / len(train)) * 100

# Sorting the results in descending order
missing_percentages = missing_percentages.sort_values(ascending=False)

# Top 30 features with most missing values
print('Top 30 features with most missing values: ')
print(missing_percentages.head(30))

Top 30 features with most missing values: 
id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
D7       93.409930
id_18    92.360721
D13      89.509263
D14      89.469469
D12      89.041047
id_04    88.768923
id_03    88.768923
D6       87.606767
id_33    87.589494
id_09    87.312290
D8       87.312290
id_10    87.312290
D9       87.312290
id_30    86.865411
id_32    86.861855
id_34    86.824771
id_14    86.445626
V138     86.123717
V139     86.123717
V148     86.123717
dtype: float64


In [12]:
# Dropping features with more than 99% of missing data
features_to_drop = missing_percentages[missing_percentages > 99].index

# Drop those columns
train = train.drop(columns=features_to_drop)
print(f"Total features dropped: {len(features_to_drop)}")
print(f"New Train dataset shape: {train.shape}")

Total features dropped: 9
New Train dataset shape: (590540, 425)


# Phase 2: Feature Engineering


In [13]:
# Converting seconds into hours, then finding the remainder of a 24-hour cycle
train['hour_of_day'] = (train['TransactionDT'] // 3600) % 24

# Converting seconds to days to find the remainder of a 7-day cycle
train['day-of-week'] = (train['TransactionDT'] // (3600*24)) % 7

print("Time features engineered successfully...")
train

Time features engineered successfully...


C:\Users\abdul\AppData\Local\Temp\ipykernel_19136\1836360001.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['hour_of_day'] = (train['TransactionDT'] // 3600) % 24
C:\Users\abdul\AppData\Local\Temp\ipykernel_19136\1836360001.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['day-of-week'] = (train['TransactionDT'] // (3600*24)) % 7


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,hour_of_day,day-of-week
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0


## Encoding and Aggregation Classification for High Cardinality Features

In [14]:
train = train.copy()

In [17]:
# Isolating only the text/categorical columns
categorical_cols = train.select_dtypes(include=['object', 'string','category']).columns

# Lists to hold automatically sorted features
target_encode_cols = []
label_encode_cols = []
drop_or_aggregate_cols = []

# Looping through and evaluating the cardinality of each
for col in categorical_cols:
    unique_count = train[col].nunique()

    if unique_count > 1000:
        # Too messy for standard encoding, flag for aggregation or dropping
        drop_or_aggregate_cols.append(col)
    elif unique_count > 100:
        # Massive cardinality, ideal for Target Encoding
        target_encode_cols.append(col)
    else:
        # Medium/Low cardinality, ideal for Label Encoding
        label_encode_cols.append(col)

# Print the resulting automated lists
print(f"Features for Target Encoding ({len(target_encode_cols)}): {target_encode_cols}")
print(f"Features for Label Encoding ({len(label_encode_cols)}): {label_encode_cols}")
print(f"Features for Aggregation/Review ({len(drop_or_aggregate_cols)}): {drop_or_aggregate_cols}")

Features for Target Encoding (2): ['id_31', 'id_33']
Features for Label Encoding (26): ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']
Features for Aggregation/Review (1): ['DeviceInfo']


## Label Encoding

In [18]:
# 1. Create an empty dictionary to hold our newly engineered columns
new_features_dict = {}

# 2. Loop through each column name in our list
for col in label_encode_cols:

    # 3. Create a dynamic name for the new column (e.g., 'DeviceInfo_label_enc')
    new_col_name = f'{col}_label_enc'

    # 4. Apply the category code transformation and save it directly into the dictionary
    new_features_dict[new_col_name] = train[col].astype('category').cat.codes

In [16]:
# Calculate the average transaction amount for each unique card1 ID
train['card1_TransactionAmt_mean'] = train.groupby('card1')['TransactionAmt'].transform('mean')

# Let's print a few columns to see the original amount vs. the historical average
print(train[['card1', 'TransactionAmt', 'card1_TransactionAmt_mean']].head(10))

   card1  TransactionAmt  card1_TransactionAmt_mean
0  13926            68.5                 351.931163
1   2755            29.0                 234.292753
2   4663            59.0                  97.015542
3  18132            50.0                 123.416340
4   4497            50.0                  96.972222
5   5937            49.0                 134.071429
6  12308           159.0                 101.880097
7  12695           422.5                 141.144645
8   2803            15.0                 142.683409
9  17399           117.0                 122.020491
